# 08 · Errores y excepciones

**Curso de Python · Bloque 2: El lenguaje**

## Objetivos

Al terminar este notebook sabrás:

1. Distinguir errores de sintaxis y excepciones, y **leer un *traceback***.
2. Capturar excepciones con `try` / `except`, y por qué hay que ser **específico**.
3. La jerarquía de excepciones de Python.
4. Usar `else`, `finally` y `with` para garantizar la limpieza.
5. Lanzar excepciones con `raise`, encadenarlas y crear las tuyas propias.
6. La filosofía **EAFP** de Python frente a **LBYL**.

Por fin vamos a explicar el `try` / `except` que venimos usando desde el primer notebook.

## Cómo trabajar con este notebook

- Haz una copia con tu nombre antes de empezar y selecciona el kernel del **`.venv`**.
- Ejecuta las celdas **en orden**. Si algo se comporta raro: *Restart* + *Run All*.
- 🧩 **Retos**: ejecuta, observa y escribe tu explicación en la celda ✍️ antes de seguir.
- 🎯 **Ejercicios**: completa la función y ejecuta la celda de comprobación (prueban varios casos).
- Dificultad: ⭐ fácil · ⭐⭐ medio · ⭐⭐⭐ para pensar un rato.

---
## 1. Errores de sintaxis y excepciones

| | Error de sintaxis | Excepción |
| --- | --- | --- |
| Cuándo | Al **leer/compilar** el código | Al **ejecutarlo** |
| ¿Se ejecuta algo? | No, nada del bloque | Sí, hasta la línea que falla |
| ¿Se puede capturar? | No (en el propio código) | Sí, con `try` / `except` |
| Ejemplo | `if x = 5:` | `int("hola")`, `1 / 0`, `lista[99]` |

In [ ]:
try:
    exec("print('antes')\nif True print('nunca')")
except SyntaxError as e:
    print(f"SyntaxError en la línea {e.lineno}: {e.msg}")

Fíjate en que ni siquiera se ha impreso `'antes'`: con un error de sintaxis no se ejecuta **nada** del bloque.

### Leer un *traceback*

Cuando una excepción no se captura, Python muestra un ***traceback***: la **pila de llamadas** que llevó hasta el error. Se lee **de abajo arriba**:

- La **última línea** dice **qué** pasó: el tipo de excepción y el mensaje.
- Las líneas de encima dicen **dónde**: cada llamada, desde la más externa (arriba) hasta la que falló (abajo).

Aquí lo mostramos con `traceback.print_exc()` para que el notebook no se detenga.

In [ ]:
import sys
import traceback


def calcular_media(valores):
    return sum(valores) / len(valores)


def informe(datos):
    media = calcular_media(datos["notas"])
    return f"Media: {media:.2f}"


try:
    print(informe({"alumno": "Ana", "notas": []}))
except ZeroDivisionError:
    traceback.print_exc(file=sys.stdout)

### 🧩 Reto 1.1 · Lee el *traceback*  ⭐

Responde mirando solo el *traceback* de la celda anterior:
1. ¿Qué tipo de error se ha producido?
2. ¿En qué función y en qué línea de código ocurrió exactamente?
3. ¿Quién llamó a esa función?
4. ¿Cuál es la **causa real** del problema y dónde la arreglarías?

✍️ **Tu respuesta:**

*(escribe aquí tu explicación)*

---
## 2. Capturar excepciones: `try` / `except`

```python
try:
    código que puede fallar
except TipoDeError as e:
    qué hacer si falla con ese tipo de error
```

- Si en el bloque `try` se lanza una excepción, Python **salta** al primer `except` cuyo tipo coincida. El resto del `try` no se ejecuta.
- Si ningún `except` coincide, la excepción **sigue propagándose** hacia arriba (a quien llamó a la función).
- `as e` da acceso al objeto excepción (mensaje, atributos…).

In [ ]:
def a_entero(texto):
    try:
        return int(texto)
    except ValueError as e:
        print(f"  No se pudo convertir {texto!r}: {e}")
        return None


print(a_entero("42"))
print(a_entero("cuarenta y dos"))

Se pueden capturar **varios tipos**, en bloques separados o en una tupla:

In [ ]:
def leer_dato(datos, clave):
    try:
        return 100 / int(datos[clave])
    except KeyError:
        return "Falta la clave"
    except (ValueError, ZeroDivisionError) as e:
        return f"Dato no válido ({type(e).__name__})"


datos = {"a": "4", "b": "cero", "c": "0"}
for clave in ["a", "b", "c", "z"]:
    print(clave, "->", leer_dato(datos, clave))

### La jerarquía de excepciones

Las excepciones son **clases** organizadas en un árbol. Un `except` captura **ese tipo y todos sus descendientes**.

```
BaseException
 ├── SystemExit, KeyboardInterrupt, GeneratorExit   ← no son "errores"
 └── Exception                                      ← base de todos los errores normales
      ├── ArithmeticError
      │    └── ZeroDivisionError, OverflowError
      ├── LookupError
      │    └── IndexError, KeyError
      ├── ValueError
      │    └── UnicodeDecodeError …
      ├── TypeError, NameError, AttributeError
      └── OSError
           └── FileNotFoundError, PermissionError, TimeoutError …
```

In [ ]:
print(ZeroDivisionError.__mro__)
print(issubclass(KeyError, LookupError))
print(issubclass(FileNotFoundError, OSError))

### 🧩 Reto 2.1 · El `except` que nunca se ejecuta  ⭐⭐

Queremos dar un mensaje específico cuando falta una clave. ¿Por qué siempre sale el mensaje genérico?

In [ ]:
config = {"puerto": 8080}

try:
    host = config["host"]
except LookupError:
    print("Error de búsqueda genérico")
except KeyError:
    print("Falta la clave 'host' en la configuración")

✍️ **Tu respuesta:**

*(escribe aquí tu explicación)*

### 🧩 Reto 2.2 · El `except` que se lo traga todo  ⭐⭐⭐

Esta función tiene un error de escritura (*typo*). Ejecútala: ¿por qué el programa dice que el precio no es válido? ¿Qué has aprendido sobre `except Exception` (o el `except:` sin tipo)?

In [ ]:
def precio_con_iva(texto):
    try:
        precio = float(texto)
        return round(precio * (1 + IVA_GENERL), 2)
    except Exception:
        return "Precio no válido"


IVA_GENERAL = 0.21
print(precio_con_iva("100"))
print(precio_con_iva("abc"))

✍️ **Tu respuesta:**

*(escribe aquí tu explicación)*

### 🎯 Ejercicio 2.3 · Conversión segura

Escribe `convertir_entero(valor, por_defecto=None)` que intente convertir `valor` a `int` y, si no es posible, devuelva `por_defecto`.

Ojo: `int()` lanza `ValueError` con textos no numéricos (`"abc"`), pero **`TypeError`** con tipos que no admite (`None`, una lista…). Ambos casos deben devolver el valor por defecto.

In [ ]:
def convertir_entero(valor, por_defecto=None):
    ...  # TODO

In [ ]:
# ✔️ Comprobación
assert convertir_entero("42") == 42
assert convertir_entero(" 7 ") == 7
assert convertir_entero(3.9) == 3
assert convertir_entero("abc") is None
assert convertir_entero("abc", 0) == 0
assert convertir_entero(None, -1) == -1
assert convertir_entero([1, 2], "x") == "x"
print("¡Correcto! 🎉")

### 🎯 Ejercicio 2.4 · División segura… pero no tanto

Escribe `dividir(a, b)` que devuelva `a / b`, o `None` si `b` es cero.

Pero **solo** debe tratar la división entre cero: si le pasan tipos incorrectos (por ejemplo, un texto), **el error debe propagarse** (es un bug de quien llama, no un caso esperado).

In [ ]:
def dividir(a, b):
    ...  # TODO

In [ ]:
# ✔️ Comprobación
assert dividir(10, 4) == 2.5
assert dividir(1, 0) is None
assert dividir(0, 5) == 0
try:
    dividir("10", 2)
    raise AssertionError("dividir('10', 2) debería lanzar TypeError, no ocultarlo")
except TypeError:
    pass
print("¡Correcto! 🎉")

---
## 3. `else`, `finally` y `with`

```python
try:
    ...          # lo que puede fallar
except ...:
    ...          # si falla
else:
    ...          # si NO ha fallado
finally:
    ...          # SIEMPRE, falle o no (incluso con return o con errores no capturados)
```

- `else` permite dejar en el `try` **solo** la línea que puede fallar (reto 2.2).
- `finally` se usa para **liberar recursos**: cerrar ficheros, conexiones, bloqueos…

In [ ]:
def procesar(texto):
    print(f"--- procesar({texto!r})")
    try:
        numero = int(texto)
    except ValueError:
        print("  except: no es un número")
    else:
        print(f"  else: el doble es {numero * 2}")
    finally:
        print("  finally: limpieza")


procesar("21")
procesar("hola")

### 🧩 Reto 3.1 · ¿Quién gana, `try` o `finally`?  ⭐⭐⭐

Predice qué devuelve cada función antes de ejecutar. ¿Por qué la segunda no devuelve `"try"`?

In [ ]:
def ejemplo_1():
    try:
        return "try"
    finally:
        print("  (se ejecuta el finally)")


def ejemplo_2():
    try:
        return "try"
    finally:
        return "finally"


def ejemplo_3():
    try:
        1 / 0
    finally:
        return "finally"


print("1:", ejemplo_1())
print("2:", ejemplo_2())
print("3:", ejemplo_3())

✍️ **Tu respuesta:**

*(escribe aquí tu explicación)*

### `with`: limpieza automática

Muchos recursos (ficheros, conexiones, bloqueos) son **gestores de contexto**: con `with`, se liberan automáticamente al salir del bloque, **aunque haya un error**. Es la forma idiomática de usarlos, y equivale a un `try` / `finally`. Lo usaremos constantemente en el tema de ficheros.

In [ ]:
import os
import tempfile

ruta = os.path.join(tempfile.gettempdir(), "prueba_curso.txt")

try:
    with open(ruta, "w", encoding="utf-8") as fichero:
        fichero.write("primera línea\n")
        1 / 0                                # error a mitad
except ZeroDivisionError:
    print("Error capturado")

print("¿Fichero cerrado?", fichero.closed)
os.remove(ruta)

---
## 4. Lanzar excepciones: `raise`

Nuestras funciones también deben lanzar excepciones cuando reciben datos que no pueden procesar. Es mucho mejor que devolver valores "especiales" como `-1` o `None` que el que llama puede olvidarse de comprobar.

Elige el tipo que mejor describa el problema:
- `TypeError`: el **tipo** del argumento es incorrecto.
- `ValueError`: el tipo es correcto pero el **valor** no es válido.

In [ ]:
def calcular_descuento(precio, porcentaje):
    if porcentaje < 0 or porcentaje > 100:
        raise ValueError(f"El porcentaje debe estar entre 0 y 100, no {porcentaje}")
    return precio * porcentaje / 100


print(calcular_descuento(200, 15))

try:
    calcular_descuento(200, 150)
except ValueError as e:
    print(f"ValueError: {e}")

Dentro de un `except`, un `raise` sin argumentos **vuelve a lanzar** la misma excepción. Útil para registrar el error y dejar que siga subiendo:

In [ ]:
def cargar_config(datos):
    try:
        return int(datos["puerto"])
    except KeyError:
        print("LOG: falta 'puerto' en la configuración")
        raise


try:
    cargar_config({})
except KeyError as e:
    print(f"El llamante recibe: KeyError {e}")

### 🧩 Reto 4.1 · Encadenar excepciones  ⭐⭐⭐

Las dos funciones convierten un error de bajo nivel en uno más descriptivo. Ejecuta y compara los dos *tracebacks*: ¿qué diferencia hay en el mensaje que une las dos excepciones? ¿Qué aporta `from e`?

In [ ]:
import sys
import traceback


def leer_puerto_v1(texto):
    try:
        return int(texto)
    except ValueError:
        raise RuntimeError("Configuración incorrecta: el puerto no es un número")


def leer_puerto_v2(texto):
    try:
        return int(texto)
    except ValueError as e:
        raise RuntimeError("Configuración incorrecta: el puerto no es un número") from e


for funcion in (leer_puerto_v1, leer_puerto_v2):
    print(f"=============== {funcion.__name__} ===============")
    try:
        funcion("ochenta")
    except RuntimeError:
        traceback.print_exc(file=sys.stdout)

✍️ **Tu respuesta:**

*(escribe aquí tu explicación)*

### Excepciones propias

Podemos crear nuestros propios tipos de excepción **heredando** de `Exception` (o de una más concreta, como `ValueError`). Todavía no hemos visto clases, pero para esto basta con esta sintaxis mínima:

In [ ]:
class SaldoInsuficienteError(ValueError):
    """El saldo no alcanza para la operación."""


try:
    raise SaldoInsuficienteError("Saldo: 50 €, importe: 80 €")
except ValueError as e:                 # también se captura como ValueError
    print(f"{type(e).__name__}: {e}")

Ventajas de tener tus propias excepciones:
- Quien usa tu código puede capturar **exactamente** tus errores, sin confundirlos con otros.
- El nombre documenta el problema (`SaldoInsuficienteError` dice más que `ValueError`).
- Al heredar de `ValueError`, el código que ya capturaba `ValueError` sigue funcionando.

### 🎯 Ejercicio 4.2 · Validar una edad

Escribe `validar_edad(edad)` que:
- Lance `TypeError` si `edad` no es un `int`. **Cuidado**: `True` y `False` son `int` (notebook 02), y tampoco deben aceptarse.
- Lance `ValueError` si está fuera del rango 0–150.
- Devuelva la edad si es válida.

Incluye un mensaje descriptivo en cada excepción.

In [ ]:
def validar_edad(edad):
    ...  # TODO

In [ ]:
# ✔️ Comprobación
assert validar_edad(0) == 0
assert validar_edad(34) == 34
assert validar_edad(150) == 150

for valor, esperado in [(-1, ValueError), (151, ValueError), ("34", TypeError), (34.0, TypeError), (True, TypeError), (None, TypeError)]:
    try:
        validar_edad(valor)
        raise AssertionError(f"validar_edad({valor!r}) debería lanzar {esperado.__name__}")
    except esperado:
        pass
print("¡Correcto! 🎉")

### 🎯 Ejercicio 4.3 · Cajero

Usando la excepción `SaldoInsuficienteError` definida arriba, escribe `retirar(saldo, importe)` que devuelva el nuevo saldo y:
- Lance `ValueError` si el importe no es positivo.
- Lance `SaldoInsuficienteError` si el importe es mayor que el saldo.

In [ ]:
def retirar(saldo, importe):
    ...  # TODO

In [ ]:
# ✔️ Comprobación
assert retirar(100, 30) == 70
assert retirar(100, 100) == 0

try:
    retirar(50, 80)
    raise AssertionError("Debería lanzar SaldoInsuficienteError")
except SaldoInsuficienteError:
    pass

try:
    retirar(50, -10)
    raise AssertionError("Debería lanzar ValueError")
except SaldoInsuficienteError:
    raise AssertionError("Un importe negativo no es saldo insuficiente: lanza ValueError")
except ValueError:
    pass
print("¡Correcto! 🎉")

---
## 5. EAFP frente a LBYL

Dos estilos para tratar los casos problemáticos:

- **LBYL** (*Look Before You Leap*, "mira antes de saltar"): comprobar antes con `if`. Habitual en C.
- **EAFP** (*Easier to Ask Forgiveness than Permission*, "es más fácil pedir perdón que permiso"): intentarlo y capturar la excepción si falla. **Es el estilo idiomático de Python.**

In [ ]:
usuario = {"nombre": "Ana"}

# LBYL
if "email" in usuario:
    email = usuario["email"]
else:
    email = "sin email"

# EAFP
try:
    email = usuario["email"]
except KeyError:
    email = "sin email"

# Y a menudo existe una tercera vía más simple:
email = usuario.get("email", "sin email")
print(email)

### 🧩 Pregunta 5.1 · ¿Por qué Python prefiere EAFP?  ⭐⭐

Piensa en este código LBYL que trabaja con un fichero. ¿Qué puede salir mal entre la comprobación y la apertura? ¿Qué otras ventajas tiene EAFP?

```python
import os
if os.path.exists("datos.csv"):
    with open("datos.csv") as f:
        ...
```

✍️ **Tu respuesta:**

*(escribe aquí tu explicación)*

### 🎯 Ejercicio 5.2 · Procesar datos sucios

Recibimos líneas con el formato `"nombre;edad"`, pero algunas vienen mal. Escribe `procesar_lineas(lineas)` que devuelva una tupla `(validos, errores)`:

- `validos`: lista de tuplas `(nombre, edad)` con la edad como `int`, solo de las líneas correctas. Usa tu `validar_edad` del ejercicio 4.2.
- `errores`: número de líneas descartadas.

Una línea es incorrecta si no tiene exactamente dos campos, si la edad no es un número o si `validar_edad` la rechaza. Quita los espacios de alrededor de cada campo con `.strip()`. **El procesamiento no debe detenerse** por una línea incorrecta.

In [ ]:
def procesar_lineas(lineas):
    ...  # TODO

In [ ]:
# ✔️ Comprobación
lineas = [
    "Ana;34",
    "Luis; 28 ",
    "Eva;cuarenta",      # edad no numérica
    "Marta;200",         # fuera de rango
    "Pedro",             # falta un campo
    "Sara;25;extra",     # campos de más
    " Iker ;0",
]
resultado = procesar_lineas(lineas)
assert resultado is not None, "procesar_lineas debe devolver una tupla (validos, errores)"
validos, errores = resultado
assert validos == [("Ana", 34), ("Luis", 28), ("Iker", 0)], f"validos = {validos}"
assert errores == 4, f"errores = {errores}"
assert procesar_lineas([]) == ([], 0)
print("¡Correcto! 🎉")

💡 Fíjate en que `nombre, edad_txt = linea.split(";")` lanza `ValueError` tanto si faltan campos como si sobran (desempaquetado del notebook 06). Por eso un único `except ValueError` cubre varios casos.

---
## 6. `assert`: comprobaciones para el programador

`assert condición, mensaje` lanza `AssertionError` si la condición es falsa. Lo hemos usado en todas las celdas de comprobación.

### 🧩 Reto 6.1 · El `assert` que desaparece  ⭐⭐⭐

Usamos `assert` para validar los datos de entrada de una función. Ejecuta la celda: el segundo script se ejecuta con `python -O`. ¿Por qué deja pasar un importe negativo?

In [ ]:
import os
import subprocess
import sys

entorno = {**os.environ, "PYTHON_COLORS": "0"}   # salida sin colores

script = """
def pagar(importe):
    assert importe > 0, "El importe debe ser positivo"
    return f"Pagados {importe} €"

print(pagar(-500))
"""

normal = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, env=entorno)
optimizado = subprocess.run([sys.executable, "-O", "-c", script], capture_output=True, text=True, env=entorno)

print("python    ->", normal.stdout.strip() or normal.stderr.strip().splitlines()[-1])
print("python -O ->", optimizado.stdout.strip() or optimizado.stderr.strip().splitlines()[-1])

✍️ **Tu respuesta:**

*(escribe aquí tu explicación)*

---
## 🏆 Reto final · Misión a Marte  🚀  ⭐⭐⭐

Controlas un *rover* en una cuadrícula de Marte de 10 × 10 casillas: `x` e `y` van de **0 a 9**. Empieza en `(0, 0)` mirando al **norte** (`"N"`). Las direcciones son `"N"` (y + 1), `"E"` (x + 1), `"S"` (y − 1) y `"O"` (x − 1).

Desde la Tierra se envían estos comandos (en texto, ¡y a veces llegan mal!):

| Comando | Efecto |
| --- | --- |
| `"A 3"` | Avanza 3 casillas en la dirección actual (el número debe ser un entero positivo) |
| `"I"` | Gira 90° a la izquierda |
| `"D"` | Gira 90° a la derecha |

### Parte 1 · Ejecutar un comando

Escribe `ejecutar(comando, x, y, direccion)` que devuelva la tupla `(x, y, direccion)` con el nuevo estado, y que:

- Lance `ComandoInvalidoError` si el comando no existe o el número no es válido. Cuando el fallo venga de `int()`, **encadena** la excepción con `raise ... from e`.
- Lance `FueraDelMapaError` si el movimiento saca al *rover* de la cuadrícula.

Las excepciones ya están definidas. Pistas: `comando.split()` y un `match` sobre la lista resultante; para girar, la cadena `"NESO"` y el operador `%` te ahorran muchos `if`.

In [ ]:
class ComandoInvalidoError(ValueError):
    """El comando recibido no se puede interpretar."""


class FueraDelMapaError(Exception):
    """El movimiento sacaría al rover de la cuadrícula."""


def ejecutar(comando, x, y, direccion):
    ...  # TODO

In [ ]:
# ✔️ Comprobación
assert ejecutar("A 3", 0, 0, "N") == (0, 3, "N")
assert ejecutar("D", 0, 0, "N") == (0, 0, "E")
assert ejecutar("I", 0, 0, "N") == (0, 0, "O")
assert ejecutar("D", 4, 4, "O") == (4, 4, "N")
assert ejecutar("A 2", 5, 5, "O") == (3, 5, "O")

for malo in ["SALTAR", "A tres", "A -1", "A 0", "A", "", "I 2"]:
    try:
        ejecutar(malo, 0, 0, "N")
        raise AssertionError(f"ejecutar({malo!r}) debería lanzar ComandoInvalidoError")
    except ComandoInvalidoError:
        pass

try:
    ejecutar("A tres", 0, 0, "N")
except ComandoInvalidoError as e:
    assert isinstance(e.__cause__, ValueError), "Encadena el error de int() con 'raise ... from e'"

try:
    ejecutar("A 1", 0, 0, "S")
    raise AssertionError("Ir al sur desde (0, 0) debería lanzar FueraDelMapaError")
except FueraDelMapaError:
    pass
print("¡Correcto! 🎉")

### Parte 2 · La misión completa

Escribe `mision(comandos)` que ejecute la lista de comandos desde `(0, 0)` mirando al norte y devuelva un diccionario con `"posicion"` (tupla), `"direccion"`, `"errores"` y `"estado"`:

- Un comando **inválido** se ignora (es ruido en la transmisión): suma 1 a `errores` y **la misión continúa**.
- Si un comando sacaría al *rover* del mapa, **la misión se aborta** (`"estado": "abortada"`) y el *rover* se queda en su última posición válida.
- Si se ejecutan todos los comandos, `"estado"` es `"completada"`.

Captura cada excepción por separado y **solo** las tuyas: un error de programación (un *typo*, por ejemplo) no debe quedar escondido (reto 2.2).

In [ ]:
def mision(comandos):
    ...  # TODO

In [ ]:
# ✔️ Comprobación
r = mision(["A 3", "D", "A 2", "bailar", "A cinco", "I", "A 1"])
assert r == {"posicion": (2, 4), "direccion": "N", "errores": 2, "estado": "completada"}, r

r = mision(["D", "A 9", "D", "A 1", "A 5"])
assert r == {"posicion": (9, 0), "direccion": "S", "errores": 0, "estado": "abortada"}, r

r = mision([])
assert r == {"posicion": (0, 0), "direccion": "N", "errores": 0, "estado": "completada"}, r
print("¡Correcto! 🎉 Houston, el rover ha llegado sano y salvo 🛰️")

💡 **Extra** ⭐⭐: inventa una ruta que lleve al *rover* desde `(0, 0)` hasta la esquina `(9, 9)` mirando al sur, con el menor número de comandos posible. ¿Quién de la clase lo consigue con menos?

✍️ **¿Qué parte te ha costado más? ¿Qué has repasado sin darte cuenta?**

*(escribe aquí tu reflexión)*

---
## ✅ Resumen

- Errores de **sintaxis**: impiden ejecutar el bloque. **Excepciones**: ocurren al ejecutar y se pueden capturar.
- El *traceback* se lee **de abajo arriba**: qué pasó y por qué cadena de llamadas.
- Captura **excepciones concretas** y mete en el `try` **solo** lo que puede fallar. **Nunca** `except:` sin tipo; `except Exception` solo en el nivel más alto, y registrando el error.
- Los `except` se evalúan **en orden**: de lo específico a lo general.
- `else` → si no hubo error. `finally` → siempre. **Nunca** `return` dentro de un `finally`.
- `with` libera los recursos automáticamente, aunque haya errores.
- `raise` para señalar datos inválidos (`TypeError` / `ValueError`). Al traducir una excepción, `raise ... from e`.
- Excepciones propias heredando de `Exception` (o de una más concreta).
- Python prefiere **EAFP** ("pedir perdón") a **LBYL** ("pedir permiso").
- `assert` es para el programador y los tests, **no** para validar datos: desaparece con `python -O`.

**Fin del bloque 2: el lenguaje.** 🎉